# DỰ BÁO ĐỘ MẶN NƯỚC BỀ MẶT BẰNG MÔ HÌNH HỌC MÁY LAI & QUÁ TRÌNH GAUSSIAN KHÔNG GIAN - THỜI GIAN
### Khắc phục triệt để Data Leakage & Tích hợp Spatio-Temporal Residual Gaussian Process
---
**Các điểm cải tiến cốt lõi so với phương pháp cũ:**
1. **Triệt tiêu Data Leakage:** Không sử dụng BMA chọn đặc trưng trên toàn bộ dataset; không chia ngẫu nhiên (Random Row Split) trên chuỗi thời gian.
2. **Phân chia dữ liệu theo Thời gian (Temporal Split):** Sử dụng dữ liệu năm 2019 đến hết tháng 02/2020 làm tập Train (~70%) và từ tháng 03/2020 đến hết tháng 06/2020 làm tập Test độc lập (~30%).
3. **Tái tạo đặc trưng thời gian chu kỳ tuần hoàn:** Giải mã 37 cột nhị phân cũ, loại bỏ 22 dòng khuyết ngày, mã hóa chu kỳ mùa vụ 365 ngày qua cặp biến `sin_doy` và `cos_doy`.
4. **Mô hình lai Không gian - Thời gian:** Dùng các mô hình Cây (Random Forest, Bagging, XGBoost, LightGBM, PSO-RF, PSO-Bagging) học quan hệ phi tuyến viễn thám, kết hợp Quá trình Gaussian (Gaussian Process) với Product Kernel `k(i, j) = k_s * k_t` để bù đắp sai số không gian - thời gian còn sót lại.

In [ ]:
# Cài đặt các thư viện bổ sung cần thiết: pyswarm, shap, lightgbm, xgboost
!pip install pyswarm shap lightgbm xgboost scikit-learn -q
print('Cài đặt thư viện hoàn tất.')


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/48.9 MB 936.7 kB/s eta 0:00:52
   ---------------------------------------- 0.5/48.9 MB 936.7 kB/s eta 0:00:52
    --------------------------------------- 0.8/48.9 MB 711.6 kB/s eta 0:01:08
    --------------------------------------- 0.8/48.9 MB 711.6 kB/s eta 0:01:08
    --------------------------------------- 1.0/48.9 MB 730.2 kB/s eta 0:01:06
    --------------------------------------- 1.0/48.9 MB 730.2 kB/s eta 0:01:06
   - -------------------------------------- 1.3/48.9 MB 702.3 kB/s eta 0:01:08
   - -------------------------------------- 1.3/48.9 MB 702.3 kB/s eta 0:01:08
   - -----


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**Cell 2: Import thư viện & Định nghĩa các hàm đo lường hiệu năng chuẩn của bài báo**
- R2 (Coefficient of Determination - Hệ số xác định)
- RMSE (Root Mean Square Error - Sai số bình phương trung bình căn, đơn vị: g/l hoặc ‰)
- MAE (Mean Absolute Error - Sai số tuyệt đối trung bình, đơn vị: g/l hoặc ‰)
- Ex_VAR (Explained Variance - Phương sai giải thích, đơn vị: %)

In [ ]:
import os
import re
import copy
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.base import clone
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import LassoCV

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ExpSineSquared, ConstantKernel, WhiteKernel
from pyswarm import pso
import shap

warnings.filterwarnings('ignore')
np.random.seed(42)

# 1. Hệ số xác định R2 (Công thức 7 bài báo)
def calc_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot) if ss_tot != 0 else 0.0

# 2. Sai số bình phương trung bình căn (RMSE - g/l) (Công thức 8 bài báo)
def calc_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

# 3. Sai số tuyệt đối trung bình (MAE - g/l) (Công thức 9 bài báo)
def calc_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

# 4. Phương sai giải thích (Ex_VAR - %) (Công thức 10 bài báo)
def calc_explained_variance(y_true, y_pred):
    var_y = np.var(y_true)
    if var_y == 0:
        return 0.0
    return (1 - np.var(y_true - y_pred) / var_y) * 100

# 5. Hàm tổng hợp 4 chỉ số theo đúng chuẩn bài báo
def evaluate_model(y_true, y_pred):
    return {
        'RMSE (‰)': round(calc_rmse(y_true, y_pred), 3),
        'MAE (‰)': round(calc_mae(y_true, y_pred), 3),
        'R2': round(calc_r2(y_true, y_pred), 3),
        'Ex_VAR (%)': round(calc_explained_variance(y_true, y_pred), 3)
    }

print('Import thư viện và định nghĩa các hàm đo lường hoàn tất.')

**Cell 3: Đọc dữ liệu thực tế, giải mã 37 cột ngày và tạo các đặc trưng chu kỳ thời gian tuần hoàn**
- Quét và lọc 37 cột ngày nhị phân cũ (`19Th0119`, `04Th0219`...).
- Phát hiện và loại bỏ 22 dòng khuyết thông tin ngày quan trắc (toàn bộ 37 cột bằng 0).
- Giải mã tên cột ra ngày chuẩn `date`, trích xuất `doy` (Day of Year: 1 - 365), `year`, `month`.
- Tạo 2 biến tuần hoàn `sin_doy` và `cos_doy` cho mô hình cây.
- Xóa sạch 37 cột nhị phân cũ để tối ưu hóa bộ nhớ và triệt tiêu nguy cơ học vẹt.

In [ ]:
# Danh sách các đường dẫn khả dĩ (Local, Kaggle, Colab)
DATA_PATHS = [
    'Check_BMA_Chuanhoa_LS8_20192020_159.csv',
    './Check_BMA_Chuanhoa_LS8_20192020_159.csv',
    '/kaggle/input/datasets/tienmanhh222/check-bma/Check_BMA_Chuanhoa_LS8_20192020_159.csv',
    '/kaggle/input/check-bma/Check_BMA_Chuanhoa_LS8_20192020_159.csv'
]

csv_file = None
for p in DATA_PATHS:
    if os.path.exists(p):
        csv_file = p
        break

if csv_file is not None:
    print(f'-> Đang đọc dữ liệu từ: {csv_file}')
    df_raw = pd.read_csv(csv_file)
else:
    raise FileNotFoundError('Không tìm thấy file CSV Check_BMA_Chuanhoa_LS8_20192020_159.csv!')

print(f'-> Kích thước dữ liệu gốc: {df_raw.shape[0]} hàng, {df_raw.shape[1]} cột.')

# 1. Tìm tất cả các cột ngày định dạng ddThmmyy
time_cols = [col for col in df_raw.columns if re.match(r'^\d{2}Th\d{4}$', col)]
print(f'-> Tìm thấy {len(time_cols)} cột ngày nhị phân.')

# 2. Hàm giải mã tên cột (ví dụ: 19Th0119 -> datetime(2019, 1, 19))
def parse_date_column(col_name):
    day = int(col_name[0:2])
    month = int(col_name[4:6])
    year = int('20' + col_name[6:8])
    return datetime(year, month, day)

# 3. Hàm trích xuất ngày an toàn (lọc bỏ các dòng toàn số 0)
def extract_safe_date(row):
    active_cols = [col for col in time_cols if row[col] == 1]
    if len(active_cols) == 1:
        return parse_date_column(active_cols[0])
    return pd.NaT

df_clean = df_raw.copy()
df_clean['date'] = df_clean.apply(extract_safe_date, axis=1)

# 4. Loại bỏ 22 dòng khuyết ngày để dữ liệu sạch 100%
num_missing = df_clean['date'].isna().sum()
df_clean = df_clean.dropna(subset=['date']).reset_index(drop=True)
print(f'-> Đã loại bỏ {num_missing} dòng khuyết ngày. Dữ liệu sạch: {df_clean.shape[0]} mẫu.')

# 5. Tạo các biến thời gian liên tục và tuần hoàn
df_clean['doy'] = df_clean['date'].dt.dayofyear
df_clean['year'] = df_clean['date'].dt.year
df_clean['month'] = df_clean['date'].dt.month
df_clean['sin_doy'] = np.sin(2 * np.pi * df_clean['doy'] / 365.25)
df_clean['cos_doy'] = np.cos(2 * np.pi * df_clean['doy'] / 365.25)

# 6. Xóa bỏ 37 cột nhị phân cũ
df_clean = df_clean.drop(columns=time_cols)

# 7. Danh sách các đặc trưng viễn thám và địa hình đầy đủ (không lọc qua BMA)
rs_features = [
    'DEM', 'Lat', 'Long', 'EVI', 'NDBI', 'NDMI', 'NDSI', 'NDVI', 'NDWI', 'VSSI', 'SAVI',
    'COSRI', 'EVI2', 'MSI', 'ND23', 'ND47', 'SI1', 'SI2', 'SI3', 'SI4', 'SI5', 'SI6',
    'SI7', 'SI8', 'SI9', 'CA', 'B', 'G', 'R', 'NIR', 'PCA1', 'PCA2', 'PCA3', 'PCA4', 'PCA5'
]

print(f'-> Bảng dữ liệu hoàn tất chuẩn bị: {df_clean.shape[0]} hàng, {df_clean.shape[1]} cột.')
display(df_clean[['date', 'year', 'month', 'doy', 'sin_doy', 'cos_doy', 'Lat', 'Long', 'SALINITY_OK']].head())

**Cell 4: Phân chia tập dữ liệu chống rò rỉ (Temporal Data Splitting - Chuẩn 70% Train / 30% Test)**
- **Tập Huấn luyện (Train Set):** Toàn bộ năm 2019 đến hết ngày 29/02/2020 (~359 mẫu, chiếm ~70.4% dữ liệu).
- **Tập Kiểm thử (Test Set):** Từ ngày 01/03/2020 đến hết tháng 06/2020 (~151 mẫu, chiếm ~29.6% dữ liệu - giai đoạn cao điểm hạn mặn lịch sử).
- **Đặc trưng mô hình Cây (`X_base`):** Toàn bộ 35 biến viễn thám & địa hình + 2 biến tuần hoàn `sin_doy`, `cos_doy` (tổng 37 biến).
- **Đặc trưng Gaussian Process (`X_st`):** Không gian `Lat`, `Long` và Thời gian `doy` (tổng 3 biến).
- **Nhãn mục tiêu (`y`):** Giữ nguyên nồng độ mặn thực tế `SALINITY_OK` (g/l hoặc ‰).

In [ ]:
target_col = 'SALINITY_OK'

# 1. Phân chia theo mốc thời gian chống Data Leakage
train_mask = (df_clean['date'] >= '2019-01-01') & (df_clean['date'] < '2020-03-01')
test_mask  = (df_clean['date'] >= '2020-03-01')

df_train = df_clean[train_mask].reset_index(drop=True)
df_test  = df_clean[test_mask].reset_index(drop=True)

# 2. Ma trận đầu vào cho các Mô hình Cây (Viễn thám + Địa hình + sin_doy + cos_doy)
base_cols = rs_features + ['sin_doy', 'cos_doy']
X_train_base = df_train[base_cols].values
X_test_base  = df_test[base_cols].values

# 3. Ma trận đầu vào cho Gaussian Process (Lat, Long, doy)
st_cols = ['Lat', 'Long', 'doy']
X_train_st = df_train[st_cols].values
X_test_st  = df_test[st_cols].values

# 4. Vector biến mục tiêu (Độ mặn thực tế)
y_train = df_train[target_col].values.astype(np.float32)
y_test  = df_test[target_col].values.astype(np.float32)

print("tien hanh loc dac trung baang LassoCV")

# Chay thuat toan loc chi tren tap Train
lasso_selector = LassoCV(cv = 5, random_state= 42, n_jobs= -1)
lasso_selector.fit(X_train_base, y_train)

# Tim cac dac trung co trong so khac 0
selected_mask = lasso_selector.coef_ != 0
selected_features = np.array(base_cols)[selected_mask]

print(f"so dac trung giu lai : {len(selected_features)} / {len(base_cols)}")
print(f"danh sach dac trung tot nhat {list(selected_features)}")

# Loc ma tran X_train va X_test
X_train_base = X_train_base[:, selected_mask]
X_test_base = X_test_base[:, selected_mask]

# 5. In thông số kiểm tra
print('=' * 80)
print('THÔNG TIN PHÂN CHIA TẬP DỮ LIỆU HUẤN LUYỆN & KIỂM THỬ (TEMPORAL SPLIT)')
print('=' * 80)
print(f'-> Tập Train (2019 đến hết T2/2020): {X_train_base.shape[0]} mẫu ({X_train_base.shape[0]/df_clean.shape[0]*100:.1f}%) | Độ mặn TB: {y_train.mean():.2f} ‰')
print(f'-> Tập Test  (T3 đến T6/2020):        {X_test_base.shape[0]} mẫu ({X_test_base.shape[0]/df_clean.shape[0]*100:.1f}%) | Độ mặn TB: {y_test.mean():.2f} ‰')
print(f'-> Số đặc trưng đưa vào mô hình Cây:  {X_train_base.shape[1]} biến ')
print(f'-> Số đặc trưng đưa vào Gaussian Process: {X_train_st.shape[1]} biến (Lat, Long, doy)')
print(f'-> Thang đo mặn thực tế: min={y_train.min():.2f}, mean={y_train.mean():.2f}, max={y_train.max():.2f} (‰)')

**Cell 5: Khởi tạo các mô hình cơ sở (Decision Tree, Bagging, Random Forest, 1D-CNN)**
- 1. Decision Tree Regressor (Cây quyết định mặc định)
- 2. Bagging Regressor cơ sở theo Table 5 bài báo (`n_estimators = 10`)
- 3. Random Forest cơ sở theo Table 5 bài báo (`n_estimators = 100`)
- 4. Mạng nơ-ron học sâu 1D-CNN (PyTorch: Conv1d -> BatchNorm -> ReLU -> Dropout -> Linear)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# 1. Decision Tree cơ sở
dt_base = DecisionTreeRegressor(random_state=42)

# 2. Bagging Regressor cơ sở theo bài báo (Table 5)
bagging_base = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=42),
    n_estimators=10,
    random_state=42,
    n_jobs=-1
)

# 3. Random Forest cơ sở theo bài báo (Table 5)
rf_base = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# 4. Mô hình mạng nơ-ron 1D-CNN (PyTorch)
class CNN1D_Model(nn.Module):
    def __init__(self, n_features=37):
        super(CNN1D_Model, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(16)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(32)
        self.relu2 = nn.ReLU()
        self.adaptive_pool = nn.AdaptiveAvgPool1d(8)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.20)
        self.fc1 = nn.Linear(32 * 8, 64)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu2(self.bn2(self.conv2(x)))
        x = self.adaptive_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

# Hàm huấn luyện mạng 1D-CNN
def train_cnn(X_tr, y_tr, X_va, y_va, epochs=200, lr=0.001, batch_size=32):
    X_tr_t = torch.FloatTensor(X_tr).unsqueeze(1)
    y_tr_t = torch.FloatTensor(y_tr).unsqueeze(1)
    X_va_t = torch.FloatTensor(X_va).unsqueeze(1)
    y_va_t = torch.FloatTensor(y_va).unsqueeze(1)
    
    train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=batch_size, shuffle=True)
    model = CNN1D_Model(n_features=X_tr.shape[1])
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    
    best_val_loss = float('inf')
    best_weights = None
    patience = 25
    patience_cnt = 0
    
    for ep in range(epochs):
        model.train()
        for bx, by in train_loader:
            optimizer.zero_grad()
            out = model(bx)
            loss = criterion(out, by)
            if torch.isfinite(loss):
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                
        model.eval()
        with torch.no_grad():
            va_pred = model(X_va_t)
            val_loss = criterion(va_pred, y_va_t).item()
            
        if np.isfinite(val_loss):
            scheduler.step(val_loss)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_weights = copy.deepcopy(model.state_dict())
                patience_cnt = 0
            else:
                patience_cnt += 1
                
        if patience_cnt >= patience:
            break
            
    if best_weights is not None:
        model.load_state_dict(best_weights)
    return model

# Tách tập con Validation từ Train phục vụ Early Stopping của 1D-CNN
X_tr_sub, X_val_sub, y_tr_sub, y_val_sub = train_test_split(
    X_train_base, y_train, test_size=0.15, random_state=42
)

print('Đã khởi tạo xong các mô hình cơ sở: Decision Tree, Bagging, Random Forest, 1D-CNN.')

**Cell 6: Tối ưu hóa siêu tham số bằng PSO (PSO-RF, PSO-Bagging, PSO-XGBoost, PSO-LightGBM)**
- Đặt cờ `USE_PRESET_OPTIMAL_PARAMS = True` để nạp ngay cấu hình tối ưu chuẩn bài báo (chạy nhanh trong 0.1 giây).
- Đặt `USE_PRESET_OPTIMAL_PARAMS = False` nếu muốn chạy lại thuật toán PSO dò tìm siêu tham số trên tập Train qua 5-Fold Cross Validation.

In [ ]:
cv_strategy = KFold(n_splits= 5, shuffle= True, random_state= 42)
from torch import legacy_contiguous_format

#Toi uu cho RF
def pso_obj_rf(params):
    n_estimators = int(round(params[0]))
    depth = int(round(params[1]))
    leaf = int(round(params[2]))
    feat = float(params[3])
    min_sample_split = int(round(params[4]))

    model = RandomForestRegressor(
        n_estimators = n_estimators,
        max_depth= depth,
        min_samples_leaf= leaf,
        max_features= feat,
        random_state= 42,
        min_samples_split= min_sample_split
    )
    scores = cross_val_score(model, X_train_base, y_train, cv = cv_strategy, scoring= 'neg_mean_squared_error', n_jobs= -1)
    return -scores.mean()

lb_rf = [120, 10, 4, 0.5, 8]
ub_rf = [250, 20, 10, 0.8, 20]

#Toi uu cho Bagging 
def pso_obj_bag(params):
    n_est = int(round(params[0]))
    depth = int(round(params[1]))
    leaf = int(round(params[2]))
    feat = float(params[3])
    min_sample_split = int(round(params[4]))
    tree = DecisionTreeRegressor(max_depth= depth, min_samples_leaf=leaf, min_samples_split= min_sample_split
    , random_state= 28)

    model = BaggingRegressor(estimator= tree, n_estimators= n_est, max_features= feat, bootstrap= True,
    random_state= 28, n_jobs= -1)
    scores = cross_val_score(model, X_train_base, y_train, cv = cv_strategy,
    scoring= 'neg_mean_squared_error', n_jobs= -1)
    return -scores.mean()

lb_bag = [120, 10, 4, 0.5, 8]
ub_bag = [250, 20, 10, 0.8, 20]

#Toi uu cho XGB
# XGBoost: [n_est, max_depth, lr, subsample, colsample, min_child_weight, gamma, reg_lambda, reg_alpha]
lb_xgb = [100, 3, 0.01, 0.6, 0.5, 1, 0.0, 0.0, 0.0]
ub_xgb = [500, 10, 0.30, 0.85, 0.8, 10, 5.0, 10.0, 5.0]
def pso_obj_xgb(p):
    model = XGBRegressor(
        n_estimators=int(round(p[0])), max_depth=int(round(p[1])),
        learning_rate=p[2], subsample=p[3], colsample_bytree=p[4],
        min_child_weight=p[5], gamma=p[6], reg_lambda=p[7], reg_alpha=p[8],
        random_state=42, n_jobs=-1)
    s = cross_val_score(model, X_train_base, y_train, cv=cv_strategy,
                        scoring='neg_mean_squared_error', n_jobs=1)
    return -s.mean()
# LightGBM: [n_est, num_leaves, lr, subsample, colsample, min_child_samples, reg_lambda, reg_alpha]
lb_lgbm = [100, 15, 0.01, 0.6, 0.5, 5,  0.0, 0.0]
ub_lgbm = [500, 100, 0.30, 1.0, 1.0, 50, 10.0, 5.0]

def pso_obj_lgbm(p):
    model = LGBMRegressor(
        n_estimators=int(round(p[0])), num_leaves=int(round(p[1])), max_depth=-1,
        learning_rate=p[2], subsample=p[3], subsample_freq=1, colsample_bytree=p[4],
        min_child_samples=int(round(p[5])), reg_lambda=p[6], reg_alpha=p[7],
        random_state=42, verbose=-1, n_jobs=-1)
    s = cross_val_score(model, X_train_base, y_train, cv=cv_strategy,
                        scoring='neg_mean_squared_error', n_jobs=1)
    return -s.mean()


In [ ]:
# CỜ ĐIỀU KHIỂN: Sử dụng ngay bộ siêu tham số tối ưu chuẩn xác
USE_PRESET_OPTIMAL_PARAMS = True

def extract_pso_result(res):
    """Trích xuất (xopt, fopt) tương thích mọi phiên bản pyswarm/scipy OptimizeResult"""
    if hasattr(res, 'x') and hasattr(res, 'fun'):
        return np.asarray(res.x, dtype=float), float(res.fun)
    elif isinstance(res, dict) and 'x' in res and 'fun' in res:
        return np.asarray(res['x'], dtype=float), float(res['fun'])
    elif isinstance(res, (tuple, list)):
        return np.asarray(res[0], dtype=float), float(res[1])
    return res, None

if USE_PRESET_OPTIMAL_PARAMS:
    print('=== SỬ DỤNG BỘ SIÊU THAM SỐ TỐI ƯU CÔNG BỐ TRONG TABLE 5 BÀI BÁO ===')
    # 1. Cấu hình PSO_Random Forest (RF_PSO trong Table 5 bài báo):
    pso_rf_model = RandomForestRegressor(
        n_estimators=250,
        max_depth=17,
        min_samples_split=2,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )
    
    # 2. Cấu hình PSO_Bagging (Bagging_PSO trong Table 5 bài báo):
    pso_bag_tree = DecisionTreeRegressor(
        max_depth=19,
        min_samples_split=2,
        min_samples_leaf=5,
        random_state=42
    )
    pso_bagging_model = BaggingRegressor(
        estimator=pso_bag_tree,
        n_estimators=259,
        random_state=42,
        n_jobs=-1
    )
    
    # 3. Cấu hình PSO_XGBoost tối ưu
    pso_xgb_model = XGBRegressor(
        n_estimators=180,
        max_depth=6,
        learning_rate=0.03,
        min_child_weight=4,
        colsample_bytree=0.7,
        subsample=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1
    )
    
    # 4. Cấu hình PSO_LightGBM tối ưu
    pso_lgbm_model = LGBMRegressor(
        n_estimators=160,
        max_depth=7,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=10,
        colsample_bytree=0.7,
        subsample=0.8,
        subsample_freq=1,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )
else:
    print('Bắt đầu chạy thuật toán PSO tìm siêu tham số trên tập Train...')
    res_rf = pso(pso_obj_rf, lb_rf, ub_rf, swarmsize=12, maxiter=10)
    xopt_rf, fopt_rf = extract_pso_result(res_rf)
    print(f"RF best MSE: {fopt_rf:.4f}")
    print(f"n_estimators : {int(round(xopt_rf[0]))}")
    print(f"max_depth: {int(round(xopt_rf[1]))}")
    print(f"min_sample_split: {int(round(xopt_rf[2]))}")
    print(f"min_sample_leaf: {int(round(xopt_rf[3]))}")

    pso_rf_model = RandomForestRegressor(
        n_estimators=int(round(xopt_rf[0])),
        max_depth=int(round(xopt_rf[1])),
        min_samples_leaf=int(round(xopt_rf[2])),
        max_features=float(xopt_rf[3]),
        min_samples_split=int(round(xopt_rf[4])),
        random_state=42,
        n_jobs=-1
    )

    res_bag = pso(pso_obj_bag, lb_bag, ub_bag, swarmsize=12, maxiter=10)
    xopt_bag, fopt_bag = extract_pso_result(res_bag)
    print(f"Bagging best MSE: {fopt_bag:.4f}")
    print(f"n_estimators : {int(round(xopt_bag[0]))}")
    print(f"max_depth: {int(round(xopt_bag[1]))}")
    print(f"min_sample_leaf : {int(round(xopt_bag[2]))}")
    print(f"max_features: {float(xopt_bag[3])}")
    print(f"min_sample_split: {int(round(xopt_bag[4]))}")

    pso_bag_tree = DecisionTreeRegressor(
        max_depth=int(round(xopt_bag[1])),
        min_samples_leaf=int(round(xopt_bag[2])),
        min_samples_split=int(round(xopt_bag[4])),
        random_state=42
    )
    pso_bagging_model = BaggingRegressor(
        estimator=pso_bag_tree,
        n_estimators=int(round(xopt_bag[0])),
        max_features=float(xopt_bag[3]),
        random_state=42,
        n_jobs=-1
    )

    res_xgb = pso(pso_obj_xgb, lb_xgb, ub_xgb, swarmsize=12, maxiter=10)
    xopt_xgb, fopt_xgb = extract_pso_result(res_xgb)
    print(f"XGB best MSE: {fopt_xgb:.4f}")
    print(f"n_estimators: {int(round(xopt_xgb[0]))}")
    print(f"max_depth: {int(round(xopt_xgb[1]))}")
    print(f"learning rate : {float(xopt_xgb[2])}")
    print(f"min_child_weight : {float(xopt_xgb[5])}")
    pso_xgb_model = XGBRegressor(
        n_estimators=int(round(xopt_xgb[0])),
        max_depth=int(round(xopt_xgb[1])),
        learning_rate=float(xopt_xgb[2]),
        subsample=float(xopt_xgb[3]),
        colsample_bytree=float(xopt_xgb[4]),
        min_child_weight=float(xopt_xgb[5]),
        gamma=float(xopt_xgb[6]),
        reg_lambda=float(xopt_xgb[7]),
        reg_alpha=float(xopt_xgb[8]),
        random_state=42,
        n_jobs=-1
    )

    res_lgbm = pso(pso_obj_lgbm, lb_lgbm, ub_lgbm, swarmsize=12, maxiter=10)
    xopt_lgbm, fopt_lgbm = extract_pso_result(res_lgbm)
    print(f"LGBM best MSE: {fopt_lgbm:.4f}")
    print(f"n_estimators : {int(round(xopt_lgbm[0]))}")
    print(f"num_leaves: {int(round(xopt_lgbm[1]))}")
    print(f"learning rate : {float(xopt_lgbm[2])}")
    print(f"min_child_samples : {int(round(xopt_lgbm[5]))}")
    pso_lgbm_model = LGBMRegressor(
        n_estimators=int(round(xopt_lgbm[0])),
        num_leaves=int(round(xopt_lgbm[1])),
        max_depth=-1,
        learning_rate=float(xopt_lgbm[2]),
        subsample=float(xopt_lgbm[3]),
        subsample_freq=1,
        colsample_bytree=float(xopt_lgbm[4]),
        min_child_samples=int(round(xopt_lgbm[5])),
        reg_lambda=float(xopt_lgbm[6]),
        reg_alpha=float(xopt_lgbm[7]),
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )

print('Đã cấu hình xong 4 mô hình tối ưu: PSO-RF, PSO-Bagging, PSO-XGBoost, PSO-LightGBM.')


**Cell 6.1: Thiết lập Quá trình Gaussian Không gian - Thời gian (Spatio-Temporal Gaussian Process)**
Mô hình hóa phần sai số (residual) theo đúng hệ thống công thức toán học từ trang vở của thầy:
- **1. Khoảng cách không gian Euclid:** `d_ij = sqrt( (lat_i - lat_j)^2 + (long_i - long_j)^2 )`
- **2. Kernel không gian (Spatial RBF Kernel):** `k_s(i, j) = exp( - d_ij^2 / (2 * l_s^2) )`
- **3. Kernel thời gian tuần hoàn (Periodic Temporal Kernel):** `k_t(i, j) = exp[ - 2 * sin^2( pi * (t_i - t_j) / 365 ) / l_t^2 ]`
- **4. Kernel tích Không - Thời gian kết hợp:** `k(i, j) = k_s(i, j) * k_t(i, j) + sigma_n^2 * I`
- **5. Dự báo tổng hợp hoàn chỉnh:** `y_hat_final = y_hat_base + e_hat_gp`

In [ ]:
import numpy as np 
from xgboost.core import _check_distributed_params
from typing import Self
from scipy.spatial.distance import cdist
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Kernel, Hyperparameter

class SpatioTemporalProductKernel(Kernel):
    def __init__(self, l_s = 0.2, l_t = 1.0, sigma_f = 1.0, l_s_bounds = (0.01, 5.0), l_t_bounds=(0.1, 5.0), sigma_f_bounds = (0.1, 50.0)):
        self.l_s = l_s
        self.l_t = l_t
        self.sigma_f = sigma_f
        self.l_s_bounds = l_s_bounds
        self.l_t_bounds = l_t_bounds
        self.sigma_f_bounds = sigma_f_bounds

    @property
    def hyperparameter_l_s(self):
        return Hyperparameter('l_s', 'numeric', self.l_s_bounds)
        #Ham Hyperparameters(): Khi custom mot kernel thi can khai bao cho sklearn biet dau la sieu tham so va ham nay nhan nhiem vu do
        # dau vao gom 3 thanh phan chinh: ten sieu tham so, kieu du lieu, khoang du lieu chay 
    @property 
    def hyperparameter_l_t(self):
        return Hyperparameter('l_t', 'numeric', self.l_t_bounds)

    @property
    def hyperparameter_sigma_f(self):
        return Hyperparameter('sigma_f', 'numeric', self.sigma_f_bounds)

    def __call__(self, X, Y = None, eval_gradient = False):
        X = np.atleast_2d(X)
        if Y is None: 
            Y = X
        X_s, Y_s = X[:, :2], Y[:, :2]
        X_t, Y_t = X[:, 2:3] , Y[:, 2:3]

        d_s_sq = cdist(X_s, Y_s, metric= 'sqeuclidean')
        k_s = np.exp(-d_s_sq / (2*(self.l_s) ** 2))

        # Metric = cityblock: la tinh theo chuan L1 |x_i - x_j|

        delta_t = cdist(X_t, Y_t, metric = "cityblock")
        sin_term = (np.sin(np.pi*delta_t)/ 365.0)**2
        k_t = np.exp(-2*sin_term/(self.l_t)**2)


        # Kernel khong - thoi gian

        K =(self.sigma_f)**2 *(k_s * k_t) 
        
        if eval_gradient:
            grad_l_s = K* (d_s_sq**2 / (self.l_s)**2) # Dao ham cua K theo l_s
            grad_l_t = K*(4*sin_term/ (self.l_t)**2) # Dao ham cua K theo l_t
            grad_sigma_f = 2.0 * K # Dao ham cua K theo sigma_f
            gradient = np.dstack((grad_l_s, grad_l_t, grad_sigma_f))
            return K, gradient

        return K

    def diag(self, X):
        return np.full(X.shape[0], self.sigma_f**2)

    def is_stationary(self):
        return True

st_kernel = SpatioTemporalProductKernel(l_s = 0.2, l_t = 1.0, sigma_f= 1.0)
gp_model_template = GaussianProcessRegressor(
    kernel= st_kernel,
    alpha = 0.5,
    n_restarts_optimizer= 2,
    random_state= 42
)
            

**Cell 7: Huấn luyện toàn bộ các Mô hình Đơn lẻ & Mô hình Lai (+ Gaussian Process)**
- Chạy tự động qua vòng lặp cho tất cả các thuật toán.
- Đã sửa lỗi typo `X_test_base` để dự báo chính xác trên tập Test (Năm 2020).
- Mô hình hóa phần dư (Residual Kriging) bằng Gaussian Process cho từng thuật toán.
- Xuất Bảng so sánh hiệu năng tổng hợp đối chuẩn (R2, RMSE, MAE, Ex_VAR).

In [ ]:
models_tree = {
    'Decsision Tree': dt_base,
    'Bagging': bagging_base,
    'PSO Bagging': pso_bagging_model,
    'Random Forest': rf_base,
    'PSO RF': pso_rf_model,
    'PSO XGB': pso_xgb_model,
    'PSO LightGBM': pso_lgbm_model
}

results = []
predictions = {}

for name, model in models_tree.items():
    print(f'Dang xu ly: {name}')
    model.fit(X_train_base, y_train) # Train model 

    pred_train_base = np.clip(model.predict(X_train_base), 0 , None)
    pred_test_base = np.clip(model.predict(X_test_base), 0 , None)
    
    # Ham danh gia model
    eval_train_base = evaluate_model(y_train, pred_train_base)
    eval_test_base = evaluate_model(y_test, pred_test_base)

    results.append({
        'Model': f"{name}",
        'Train_RMSE (‰)': eval_train_base['RMSE (‰)'],
        'Train_MAE (‰)' : eval_train_base['MAE (‰)'],
        'Train_R2'      : eval_train_base['R2'],
        'Train_Ex_VAR (%)': eval_train_base['Ex_VAR (%)'],
        'Test_RMSE (‰)': eval_test_base['RMSE (‰)'],
        'Test_MAE (‰)': eval_test_base['MAE (‰)'],
        'Test_R2': eval_test_base['R2'],
        'Test_Ex_VAR (%)': eval_test_base['Ex_VAR (%)']
    })

    # Tinh phan du sai so tren tap Train
    residual_train = y_train - pred_train_base

    # Huan luyen GP tren phan du sai so 
    gp = clone(gp_model_template)
    gp.fit(X_train_st, residual_train)

    # Du bao cua GP tren tap test 
    pred_gp_test = gp.predict(X_test_st)
    
    # Du bao tong hop (GP + cac Model)
    pred_test_hybrid = np.clip(pred_gp_test + pred_test_base, 0, None)
    eval_test_hybrid = evaluate_model(y_test, pred_test_hybrid)

    # Luu du doan de ve bieu do 
    predictions[name] = {
        'base' : pred_test_base,
        'hybrid' : pred_test_hybrid
    }

    results.append({
        'Model': f"{name} + GP",
        'Train_RMSE (‰)': eval_train_base['RMSE (‰)'],
        'Train_MAE (‰)' : eval_train_base['MAE (‰)'],
        'Train_R2'      : eval_train_base['R2'],
        'Train_Ex_VAR (%)': eval_train_base['Ex_VAR (%)'],
        'Test_RMSE (‰)': eval_test_hybrid['RMSE (‰)'],
        'Test_MAE (‰)': eval_test_hybrid['MAE (‰)'],
        'Test_R2': eval_test_hybrid['R2'],
        'Test_Ex_VAR (%)': eval_test_hybrid['Ex_VAR (%)']
    })

results_df = pd.DataFrame(results).set_index('Model')
print("\n Bang tong hop hieu nang")
display(results_df)

In [ ]:
# ----------------------------------------------------------------------
# 2. HUẤN LUYỆN VÀ ĐÁNH GIÁ 1D-CNN (PyTorch)
# ----------------------------------------------------------------------
print('\n-> Đang huấn luyện: 1D-CNN (PyTorch)...')
cnn_model = train_cnn(X_tr_sub, y_tr_sub, X_val_sub, y_val_sub, epochs=200, lr=0.001, batch_size=32)

X_train_tensor = torch.FloatTensor(X_train_base).unsqueeze(1)
X_test_tensor  = torch.FloatTensor(X_test_base).unsqueeze(1)

cnn_model.eval()
with torch.no_grad():
    cnn_pred_train = np.clip(cnn_model(X_train_tensor).numpy().flatten(), 0, None)
    cnn_pred_test  = np.clip(cnn_model(X_test_tensor).numpy().flatten(), 0, None)

cnn_eval_tr = evaluate_model(y_train, cnn_pred_train)
cnn_eval_te = evaluate_model(y_test, cnn_pred_test)

# Đơn lẻ 1D-CNN
results.append({
    'Model': '1D-CNN (Đơn lẻ)',
    'Type': 'Base',
    'Train_RMSE (‰)': cnn_eval_tr['RMSE (‰)'],
    'Train_R2': cnn_eval_tr['R2'],
    'Test_RMSE (‰)': cnn_eval_te['RMSE (‰)'],
    'Test_MAE (‰)': cnn_eval_te['MAE (‰)'],
    'Test_R2': cnn_eval_te['R2'],
    'Test_Ex_VAR (%)': cnn_eval_te['Ex_VAR (%)']
})

# Lai 1D-CNN + GP
cnn_residual_tr = y_train - cnn_pred_train
cnn_gp = clone(gp_model_template)
cnn_gp.fit(X_train_st, cnn_residual_tr)
cnn_hybrid_te = np.clip(cnn_pred_test + cnn_gp.predict(X_test_st), 0, None)
cnn_eval_hy = evaluate_model(y_test, cnn_hybrid_te)

results.append({
    'Model': '1D-CNN + GP (Mô hình lai)',
    'Type': 'Hybrid',
    'Train_RMSE (‰)': cnn_eval_tr['RMSE (‰)'],
    'Train_R2': cnn_eval_tr['R2'],
    'Test_RMSE (‰)': cnn_eval_hy['RMSE (‰)'],
    'Test_MAE (‰)': cnn_eval_hy['MAE (‰)'],
    'Test_R2': cnn_eval_hy['R2'],
    'Test_Ex_VAR (%)': cnn_eval_hy['Ex_VAR (%)']
})

predictions['1D-CNN'] = {'base': cnn_pred_test, 'hybrid': cnn_hybrid_te}
print(f'   [1D-CNN Đơn lẻ] Test R2 = {cnn_eval_te["R2"]:.3f} | Test RMSE = {cnn_eval_te["RMSE (‰)"]:.3f} ‰')
print(f'   [1D-CNN + GP]   Test R2 = {cnn_eval_hy["R2"]:.3f} | Test RMSE = {cnn_eval_hy["RMSE (‰)"]:.3f} ‰')


In [ ]:

# ----------------------------------------------------------------------
# 3. TỔNG HỢP VÀ HIỂN THỊ BẢNG KẾT QUẢ ĐỐI CHUẨN TOÀN DIỆN
# ----------------------------------------------------------------------
results_df = pd.DataFrame(results).set_index('Model')
print('\n' + '=' * 95)
print('BẢNG SO SÁNH HIỆU NĂNG TỔNG HỢP TRƯỚC VÀ SAU KHI TÍCH HỢP GAUSSIAN PROCESS')
print('=' * 95)
display(results_df)

results_df.to_csv('result_hybrid_gp.csv')
print('-> Đã lưu bảng kết quả hoàn chỉnh vào file: result_hybrid_gp.csv')

**Cell 8: Trực quan hóa kết quả dự báo (Scatter Plot 1:1 & Phân tích sai số Residual)**
- Biểu đồ Scatter Plot phân tán (Observed vs Predicted) so sánh trực tiếp mô hình Đơn lẻ vs Mô hình Lai (+ GP).
- Đánh giá trực quan mức độ cải thiện và khả năng hội tụ dọc theo đường chéo 1:1.

In [ ]:
# Chọn mô hình tốt nhất để trực quan hóa (ví dụ: 'PSO_Random Forest' hoặc 'PSO_LightGBM')
target_model = 'PSO_Random Forest'

pred_te_base   = predictions[target_model]['base']
pred_te_hybrid = predictions[target_model]['hybrid']

eval_b = evaluate_model(y_test, pred_te_base)
eval_h = evaluate_model(y_test, pred_te_hybrid)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
max_val = max(y_test.max(), pred_te_hybrid.max(), pred_te_base.max()) + 2

# Subplot 1: Mô hình đơn lẻ
axes[0].scatter(y_test, pred_te_base, alpha=0.6, edgecolors='k', linewidths=0.5, c='#1f77b4')
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='1:1 Line')
axes[0].set_xlabel('Observed Salinity (‰)', fontsize=12)
axes[0].set_ylabel('Predicted Salinity (‰)', fontsize=12)
axes[0].set_title(f'(a) {target_model} (Đơn lẻ)\nRMSE = {eval_b["RMSE (‰)"]} ‰, MAE = {eval_b["MAE (‰)"]} ‰, R² = {eval_b["R2"]}', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, linestyle=':', alpha=0.6)

# Subplot 2: Mô hình lai + Gaussian Process
axes[1].scatter(y_test, pred_te_hybrid, alpha=0.6, edgecolors='k', linewidths=0.5, c='#2ca02c')
axes[1].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='1:1 Line')
axes[1].set_xlabel('Observed Salinity (‰)', fontsize=12)
axes[1].set_ylabel('Predicted Salinity (‰)', fontsize=12)
axes[1].set_title(f'(b) {target_model} + GP (Mô hình lai)\nRMSE = {eval_h["RMSE (‰)"]} ‰, MAE = {eval_h["MAE (‰)"]} ‰, R² = {eval_h["R2"]}', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.savefig('scatter_plot_hybrid_gp_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('-> Đã lưu biểu đồ so sánh: scatter_plot_hybrid_gp_comparison.png')